In [7]:
import pandas as pd
import numpy as np

In [8]:
df = pd.read_csv("/kaggle/input/datasets/mdashiqurrahman6756/student-messy/students_messy.csv")
raw = df.copy()          # keep an untouched copy so we can compare at the end
print("Shape (rows, columns):", df.shape)
df.head(8)

Shape (rows, columns): (22, 12)


,Student ID,Student Name,gender,Age,Class,Math Marks,English Marks,Science_Marks,Attendance %,Email,Admission Date,City
0,S001,rahim uddin,Male,15,Nine,78,65.0,82,92%,rahim.uddin@gmail.com,2023-01-15,Dhaka
1,S002,KARIM AHMED,M,16,Ten,85,NaN,90,88%,karim.ahmed@yahoo.com,15/02/2023,dhaka
2,S003,Fatema Begum,female,15,Nine,92,88.0,95,95%,fatema.b@gmail.com,2023-03-10,Chittagong
3,S004,nusrat jahan,F,14,Eight,67,72.0,NaN,80%,nusrat.jahan@gmail,2023/04/05,Chattogram
4,S005,Tanvir Hasan,Male,fifteen,Nine,55,60.0,58,75%,tanvir.h@gmail.com,2023-01-20,Sylhet
5,S006,Sadia Islam,Female,16,Ten,150,85.0,88,90%,sadia.islam@gmail.com,2023-02-28,Dhaka
6,S002,KARIM AHMED,M,16,Ten,85,NaN,90,88%,karim.ahmed@yahoo.com,15/02/2023,dhaka
7,S007,Imran Khan,male,15,nine,70,68.0,74,85%,NaN,2023-03-15,Rajshahi


In [9]:
df.isna().sum()

Student ID        0
 Student Name     1
gender            0
Age               1
Class             0
 Math Marks       0
English Marks     4
Science_Marks     2
Attendance %      1
Email             1
Admission Date    0
City              0
dtype: int64

In [10]:
print("Before:", list(df.columns))

df.columns = (
    df.columns
      .str.strip()                 # remove spaces at the edges
      .str.lower()                 # lowercase everything
      .str.replace("%", "pct")     # replace the symbol with a word
      .str.replace(" ", "_")       # spaces -> underscores
)

print("After: ", list(df.columns))

Before: ['Student ID ', ' Student Name', 'gender', 'Age', 'Class', ' Math Marks ', 'English Marks', 'Science_Marks', 'Attendance %', 'Email', 'Admission Date', 'City']
After:  ['student_id', 'student_name', 'gender', 'age', 'class', 'math_marks', 'english_marks', 'science_marks', 'attendance_pct', 'email', 'admission_date', 'city']


In [11]:
print("Number of fully duplicated rows:", df.duplicated().sum())
df[df.duplicated(keep=False)]     # keep=False shows ALL copies, not just the extras

Number of fully duplicated rows: 2


,student_id,student_name,gender,age,class,math_marks,english_marks,science_marks,attendance_pct,email,admission_date,city
1,S002,KARIM AHMED,M,16,Ten,85,NaN,90,88%,karim.ahmed@yahoo.com,15/02/2023,dhaka
6,S002,KARIM AHMED,M,16,Ten,85,NaN,90,88%,karim.ahmed@yahoo.com,15/02/2023,dhaka
9,S009,Arif Hossain,Male,16,Ten,-5,70.0,65,70%,arif.h@gmail.com,2023-01-25,Dhaka
16,S009,Arif Hossain,Male,16,Ten,-5,70.0,65,70%,arif.h@gmail.com,2023-01-25,Dhaka


In [15]:
df = df.drop_duplicates().reset_index(drop=True)
print("Rows after removing duplicates:", len(df))

# Also check: is student_id now unique? (IDs should never repeat)
print("Any repeated student_id left?", df["student_id"].duplicated().any())

Rows after removing duplicates: 20
Any repeated student_id left? False


In [16]:
text_cols = ["student_id", "student_name", "gender", "class", "email", "city"]

for col in text_cols:
    df[col] = df[col].str.strip()

df["student_name"] = df["student_name"].str.title()
df["city"]         = df["city"].str.title()
df["class"]        = df["class"].str.title()
df["email"]        = df["email"].str.lower()

df[["student_name", "class", "city"]].head(10)

,student_name,class,city
0,Rahim Uddin,Nine,Dhaka
1,Karim Ahmed,Ten,Dhaka
2,Fatema Begum,Nine,Chittagong
3,Nusrat Jahan,Eight,Chattogram
4,Tanvir Hasan,Nine,Sylhet
5,Sadia Islam,Ten,Dhaka
6,Imran Khan,Nine,Rajshahi
7,Mitu Akter,Eight,Khulna
8,Arif Hossain,Ten,Dhaka
9,Sumaiya Rahman,Nine,Sylhet


In [17]:
print(df["gender"].value_counts(), "\n")
print(df["city"].value_counts())

gender
Male      6
Female    4
M         3
female    3
F         3
male      1
Name: count, dtype: int64 

city
Dhaka         6
Sylhet        3
Khulna        3
Chittagong    2
Chattogram    2
Rajshahi      2
Barishal      1
Barisal       1
Name: count, dtype: int64


In [18]:
gender_map = {"male": "Male", "m": "Male", "female": "Female", "f": "Female"}
df["gender"] = df["gender"].str.lower().map(gender_map)

city_map = {"Chittagong": "Chattogram", "Barisal": "Barishal"}
df["city"] = df["city"].replace(city_map)

print(df["gender"].value_counts(), "\n")
print(df["city"].value_counts())

gender
Male      10
Female    10
Name: count, dtype: int64 

city
Dhaka         6
Chattogram    4
Sylhet        3
Khulna        3
Rajshahi      2
Barishal      2
Name: count, dtype: int64


In [19]:
print("Unmapped genders:", df["gender"].isna().sum())

Unmapped genders: 0


In [20]:
for col in ["age", "math_marks", "english_marks", "science_marks"]:
    converted = pd.to_numeric(df[col], errors="coerce")
    bad = df.loc[converted.isna() & df[col].notna(), col]
    print(f"{col:15s} -> non-numeric values: {list(bad)}")

age             -> non-numeric values: ['fifteen']
math_marks      -> non-numeric values: []
english_marks   -> non-numeric values: []
science_marks   -> non-numeric values: ['missing']


In [21]:
df["age"] = df["age"].replace({"fifteen": "15"})

for col in ["age", "math_marks", "english_marks", "science_marks"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Attendance: remove the % sign first, then convert
df["attendance_pct"] = pd.to_numeric(df["attendance_pct"].str.replace("%", ""), errors="coerce")

df[["age", "math_marks", "english_marks", "science_marks", "attendance_pct"]].dtypes

age               float64
math_marks          int64
english_marks     float64
science_marks     float64
attendance_pct    float64
dtype: object

In [22]:
print("Before:", df["admission_date"].unique()[:8])

formats = ["%Y-%m-%d", "%Y/%m/%d", "%d/%m/%Y", "%d-%m-%Y"]

parsed = pd.Series(pd.NaT, index=df.index)
for fmt in formats:
    attempt = pd.to_datetime(df["admission_date"], format=fmt, errors="coerce")
    parsed = parsed.fillna(attempt)       # only fill rows not already parsed

df["admission_date"] = parsed
print("Unparsed dates:", df["admission_date"].isna().sum())
df[["student_id", "admission_date"]].head(12)

Before: ['2023-01-15' '15/02/2023' '2023-03-10' '2023/04/05' '2023-01-20'
 '2023-02-28' '2023-03-15' '2023-04-12']
Unparsed dates: 0


,student_id,admission_date
0,S001,2023-01-15
1,S002,2023-02-15
2,S003,2023-03-10
3,S004,2023-04-05
4,S005,2023-01-20
5,S006,2023-02-28
6,S007,2023-03-15
7,S008,2023-04-12
8,S009,2023-01-25
9,S010,2023-02-10


In [23]:
rules = {
    "age": (10, 20),
    "math_marks": (0, 100),
    "english_marks": (0, 100),
    "science_marks": (0, 100),
    "attendance_pct": (0, 100),
}

for col, (low, high) in rules.items():
    bad = df[(df[col] < low) | (df[col] > high)]
    if len(bad):
        print(f"{col}: invalid ->", dict(zip(bad["student_id"], bad[col])))

age: invalid -> {'S008': 200.0}
math_marks: invalid -> {'S006': 150, 'S009': -5}
attendance_pct: invalid -> {'S017': 110.0}


In [24]:
for col, (low, high) in rules.items():
    df.loc[(df[col] < low) | (df[col] > high), col] = np.nan

df[list(rules)].describe().loc[["min", "max"]]

,age,math_marks,english_marks,science_marks,attendance_pct
min,14.0,55.0,58.0,58.0,70.0
max,16.0,95.0,93.0,97.0,99.0


In [25]:
pattern = r"^[\w.+-]+@[\w-]+\.[\w.]+$"
valid = df["email"].str.match(pattern, na=False)

print("Invalid emails:")
print(df.loc[~valid & df["email"].notna(), ["student_id", "email"]])

df.loc[~valid, "email"] = np.nan

Invalid emails:
   student_id                email
3        S004   nusrat.jahan@gmail
15       S016  riya.das@@gmail.com


In [26]:
# 1. Drop rows with no name
df = df.dropna(subset=["student_name"]).reset_index(drop=True)

# 2. Age -> median age of the same class
df["age"] = df["age"].fillna(df.groupby("class")["age"].transform("median"))

# 3. Marks -> flag first, THEN fill (order matters!)
mark_cols = ["math_marks", "english_marks", "science_marks"]
df["marks_estimated"] = df[mark_cols].isna().any(axis=1)
for col in mark_cols:
    df[col] = df[col].fillna(df[col].median())

# 4. Attendance -> median
df["attendance_pct"] = df["attendance_pct"].fillna(df["attendance_pct"].median())

df.isna().sum()

student_id         0
student_name       0
gender             0
age                0
class              0
math_marks         0
english_marks      0
science_marks      0
attendance_pct     0
email              3
admission_date     0
city               0
marks_estimated    0
dtype: int64

In [27]:
df["age"] = df["age"].astype(int)
df["gender"] = df["gender"].astype("category")
df["class"] = df["class"].astype("category")

# Automatic quality checklist
assert df["student_id"].is_unique,                     "Duplicate IDs!"
assert df["age"].between(10, 20).all(),                "Bad ages!"
assert df[mark_cols].apply(lambda s: s.between(0, 100)).all().all(), "Bad marks!"
assert df["attendance_pct"].between(0, 100).all(),     "Bad attendance!"
assert set(df["gender"].unique()) <= {"Male", "Female"}, "Bad gender values!"
assert df["admission_date"].notna().all(),             "Bad dates!"
print("✅ All checks passed")

df.to_csv("students_clean.csv", index=False)
df.info()

✅ All checks passed
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19 entries, 0 to 18
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   student_id       19 non-null     object        
 1   student_name     19 non-null     object        
 2   gender           19 non-null     category      
 3   age              19 non-null     int64         
 4   class            19 non-null     category      
 5   math_marks       19 non-null     float64       
 6   english_marks    19 non-null     float64       
 7   science_marks    19 non-null     float64       
 8   attendance_pct   19 non-null     float64       
 9   email            16 non-null     object        
 10  admission_date   19 non-null     datetime64[ns]
 11  city             19 non-null     object        
 12  marks_estimated  19 non-null     bool          
dtypes: bool(1), category(2), datetime64[ns](1), float64(4), int64(1), object(4)
m

In [28]:
df

,student_id,student_name,gender,age,class,math_marks,english_marks,science_marks,attendance_pct,email,admission_date,city,marks_estimated
0,S001,Rahim Uddin,Male,15,Nine,78.0,65.0,82.0,92.0,rahim.uddin@gmail.com,2023-01-15,Dhaka,False
1,S002,Karim Ahmed,Male,16,Ten,85.0,76.5,90.0,88.0,karim.ahmed@yahoo.com,2023-02-15,Dhaka,True
2,S003,Fatema Begum,Female,15,Nine,92.0,88.0,95.0,95.0,fatema.b@gmail.com,2023-03-10,Chattogram,False
3,S004,Nusrat Jahan,Female,14,Eight,67.0,72.0,83.0,80.0,NaN,2023-04-05,Chattogram,True
4,S005,Tanvir Hasan,Male,15,Nine,55.0,60.0,58.0,75.0,tanvir.h@gmail.com,2023-01-20,Sylhet,False
5,S006,Sadia Islam,Female,16,Ten,79.0,85.0,88.0,90.0,sadia.islam@gmail.com,2023-02-28,Dhaka,True
6,S007,Imran Khan,Male,15,Nine,70.0,68.0,74.0,85.0,NaN,2023-03-15,Rajshahi,False
7,S008,Mitu Akter,Female,14,Eight,81.0,79.0,84.0,97.0,mitu.akter@gmail.com,2023-04-12,Khulna,False
8,S009,Arif Hossain,Male,16,Ten,79.0,70.0,65.0,70.0,arif.h@gmail.com,2023-01-25,Dhaka,True
9,S010,Sumaiya Rahman,Female,15,Nine,88.0,91.0,87.0,93.0,sumaiya.r@gmail.com,2023-02-10,Sylhet,False
